# 42 — Uncertainty & ranking stability

Calendar-week block bootstrap for uncertainty plus deterministic handling of isotonic score ties.


In [1]:
from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec
from argos.config.modeling import load_modeling_spec
from argos.config.robustness import load_robustness_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()
(paths.root / "reports" / "robustness").mkdir(parents=True, exist_ok=True)
(paths.root / "data" / "m5").mkdir(parents=True, exist_ok=True)
(paths.root / "artifacts" / "models" / "m5").mkdir(parents=True, exist_ok=True)

m4_gate = paths.docs / "decisions" / "M4_GATE.md"
assert m4_gate.exists() and "M4 ✅ CLOSED" in m4_gate.read_text(encoding="utf-8"), (
    "M4 no está formalmente cerrado. No iniciar M5."
)
GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")
MODEL_SPEC = load_modeling_spec(paths.configs / "modeling.toml")
M5_SPEC = load_robustness_spec(paths.configs / "robustness.toml")
MANIFEST_DIR = paths.root / "artifacts" / "manifests"

assert GOLD_SPEC.schema_version == M5_SPEC.core.benchmark_gold_schema
assert MODEL_SPEC.schema_version == M5_SPEC.core.benchmark_model_schema
{
    "gold_schema": GOLD_SPEC.schema_version,
    "model_schema": MODEL_SPEC.schema_version,
    "m5_schema": M5_SPEC.schema_version,
    "benchmark_model": M5_SPEC.core.benchmark_model_name,
    "benchmark_calibration": M5_SPEC.core.benchmark_calibration,
    "consumed_test_year": M5_SPEC.governance.consumed_test_year,
    "fresh_oot_year": M5_SPEC.governance.fresh_oot_year,
}

assert (paths.reports / "robustness" / "41_validation.parquet").exists(), "Ejecuta 41 primero."
from argos.robustness.uncertainty import BUILD_IMPLEMENTATION
BUILD_IMPLEMENTATION


'm5-42-block-bootstrap-ranking-stability-v1'

In [2]:
from argos.robustness.uncertainty import build_uncertainty_and_ranking_stability

result = build_uncertainty_and_ranking_stability(
    reports_root=paths.reports,
    manifest_dir=MANIFEST_DIR,
    modeling_spec=MODEL_SPEC,
    robustness_spec=M5_SPEC,
)
result.intervals


,metric,point_estimate_sample,lower,upper,confidence_level,bootstrap_method,replicates
0,pr_auc,0.039403,0.024778,0.058328,0.95,calendar-week block bootstrap,200
1,roc_auc,0.861939,0.819733,0.893022,0.95,calendar-week block bootstrap,200
2,brier,0.003878,0.002861,0.005310,0.95,calendar-week block bootstrap,200
3,ece,0.002096,0.001131,0.002799,0.95,calendar-week block bootstrap,200
4,precision_at_k,0.071000,0.042973,0.099694,0.95,calendar-week block bootstrap,200
5,recall_at_k,0.179293,0.121322,0.229521,0.95,calendar-week block bootstrap,200
6,lift_at_k,17.929293,12.129758,22.939971,0.95,calendar-week block bootstrap,200


In [3]:
result.tie_sensitivity


,review_fraction,reviewed,top_k_jaccard,deterministic_captured,legacy_captured,captured_delta,deterministic_precision,legacy_precision,deterministic_recall,legacy_recall,deterministic_lift,legacy_lift
0,0.001,429,0.100000,6,25,-19,0.013986,0.058275,0.003827,0.015944,3.818289,15.909537
1,0.005,2141,0.833833,97,111,-14,0.045306,0.051845,0.061862,0.070791,12.368866,14.154063
2,0.010,4281,0.894667,271,267,4,0.063303,0.062369,0.172832,0.170281,17.282194,17.027107
3,0.025,10702,0.914148,572,567,5,0.053448,0.052981,0.364796,0.361607,14.591700,14.464151
4,0.050,21404,0.883989,800,797,3,0.037376,0.037236,0.510204,0.508291,10.203986,10.165721
5,0.100,42808,0.971039,1055,1053,2,0.024645,0.024598,0.672832,0.671556,6.728253,6.715498


In [4]:
result.deterministic_ranking


,model,split,review_fraction,reviewed,captured_positives,precision_at_k,recall_at_k,lift_at_k,cumulative_gain,tie_break
0,ARGOS_V1_deterministic_tiebreak,validation_selection,0.001,429,6,0.013986,0.003827,3.818289,0.003827,score_desc/raw_score_desc/contract_id_asc
1,ARGOS_V1_deterministic_tiebreak,validation_selection,0.005,2141,97,0.045306,0.061862,12.368866,0.061862,score_desc/raw_score_desc/contract_id_asc
2,ARGOS_V1_deterministic_tiebreak,validation_selection,0.010,4281,271,0.063303,0.172832,17.282194,0.172832,score_desc/raw_score_desc/contract_id_asc
3,ARGOS_V1_deterministic_tiebreak,validation_selection,0.025,10702,572,0.053448,0.364796,14.591700,0.364796,score_desc/raw_score_desc/contract_id_asc
4,ARGOS_V1_deterministic_tiebreak,validation_selection,0.050,21404,800,0.037376,0.510204,10.203986,0.510204,score_desc/raw_score_desc/contract_id_asc
5,ARGOS_V1_deterministic_tiebreak,validation_selection,0.100,42808,1055,0.024645,0.672832,6.728253,0.672832,score_desc/raw_score_desc/contract_id_asc


In [5]:
result.validation


,check_id,category,blocking,passed,observed,requirement
0,uncertainty_temporal_blocks_sufficient,uncertainty,True,True,"{""valid_replicates"": 200, ""weeks"": 27}",uncertainty uses multiple temporal blocks and ...
1,ranking_tie_break_frozen,ranking,True,True,"{""tie_break"": [""score_desc"", ""raw_score_desc"",...",operational ranking has an explicit determinis...
2,operational_lift_ci_above_random,usefulness,True,True,"{""fraction"": 0.01, ""lower"": 12.129757903180861...",the confidence interval for Lift@K at the oper...
3,tie_sensitivity_quantified,ranking,True,True,"{""captured_delta"": 4, ""operational_top_k_jacca...",M5 quantifies the impact of isotonic score tie...
4,uncertainty_no_test_selection,governance,True,True,"{""consumed_test_year"": 2025, ""period_end"": ""20...",uncertainty/tie policy is frozen from developm...


In [6]:
blocking = result.validation.loc[result.validation["blocking"]]
print("42 ✅ PASS" if result.ready_for_next_notebook else "42 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Manifest:", result.manifest_path)
assert result.ready_for_next_notebook


42 ✅ PASS
Blocking checks: 5/5
Manifest: /Users/macm1pro/Desktop/projects/argos/argos-code/artifacts/manifests/m5_uncertainty_ranking.json
